# 9 : Feuille de TD (Exercices)

Cette feuille de travaux dirigés (*Problem Set*) a pour objectif de valider votre maîtrise de l'arithmétique bivariée and de la normalisation des écarts avant le passage à l'automatisation logicielle. Les calculs doivent être résolus manuellement à l'aide des formules théoriques vues dans les sections précédentes.

### Exercice 1 : Calcul de la métrique RSS et caractérisation géométrique

On étudie un panier composé de deux crypto-actifs fortement liés, la variable dépendante Y and la variable explicative X. Un modèle de régression linéaire bivarié a extrait les paramètres structurels suivants pour la juste valeur : $\hat{a} = 0.5$ and $\hat{b} = 1.2$. 

Votre base de données renvoie les trois observations historiques réelles suivantes :
*   $t=1$ : $x_1 = 2, y_1 = 3$
*   $t=2$ : $x_2 = 4, y_2 = 5$
*   $t=3$ : $x_3 = 5, y_3 = 6$

1.  Calculer la valeur théorique prédite $\hat{y}_t$ pour chaque point de l'échantillon.
2.  Isoler le vecteur des résidus empiriques $e_t$ pour les trois instants temporels.
3.  Calculer la valeur numérique de la Somme des Carrés des Résidus (RSS) du modèle.



In [11]:
import numpy as np

def estimator(x: float) -> float:
    return (0.5 + 1.2 * x)

y_true = np.array([3, 5, 6])
y_pred = np.array(list(map(estimator, [2, 4, 5])))
y_pred

array([2.9, 5.3, 6.5])

In [15]:
error = y_true - y_pred
error

array([ 0.1, -0.3, -0.5])

In [19]:
error_sq = error * error
error_sq

array([0.01, 0.09, 0.25])

In [20]:
sum(error_sq)

0.3499999999999999

In [22]:
import numpy as np

# Données d'entrée (X majuscule si matrice, x minuscule si vecteur simple)
x = np.array([2.0, 4.0, 5.0])
y_true = np.array([3.0, 5.0, 6.0])

# Paramètres du modèle (avec tiret bas si estimés automatiquement)
intercept = 0.5
coef = 1.2

# Calculs et métriques
y_pred = intercept + coef * x
residuals = y_true - y_pred
rss = np.sum(residuals**2)

print(f"Vecteur des résidus : {residuals}")
print(f"RSS brut : {rss}")
print(f"RSS arrondi pour la console : {np.round(rss, 4)}")


Vecteur des résidus : [ 0.1 -0.3 -0.5]
RSS brut : 0.3499999999999999
RSS arrondi pour la console : 0.35


### Exercice 2 : Génération de signaux d'arbitrage par le Z-Score du spread

Dans le cadre d'une stratégie de retour à la moyenne sur le Forex, un modèle estime la relation d'équilibre horaire entre l'AUD/USD (Y) and le NZD/USD (X). L'équation de la droite d'équilibre est définie par :
$$\hat{y}_t = -0.02 + 0.90 x_t$$

Après analyse statistique de l'historique sur les 500 dernières bougies, l'écart-type de la distribution des résidus de ce modèle a été calculé and stabilisé à $\sigma_e = 0.0050$. 

À une heure de cotation précise t, les flux de marché affichent les cours réels suivants sur votre terminal de trading :
*   Cours réel du NZD/USD ($x_t$) = 0.6000
*   Cours réel de l'AUD/USD ($y_t$) = 0.5350

1.  Calculer la valeur théorique ($\hat{y}_t$) induite par la droite d'équilibre du modèle.
2.  Calculer la valeur brute du résidu empirique ($e_t$) à cet instant précis.
3.  Calculer le Z-Score ($Z_t$) de cette anomalie géométrique.
4.  En déduire la règle d'engagement opérationnelle que le moteur du robot de trading doit déclencher (Neutre, Vente synthétique ou Achat synthétique). Justifier la réponse.

In [35]:

print('--- EXERCICE 2 -----')

intercept = -0.02
coef = 0.90

x = 0.6
y_pred = intercept + coef * 0.6
print(f'y_pred = {y_pred}')

y_true = 0.5350
residual = y_true - y_pred
print(f'residual = {np.round(residual,4)}')

sigma = 0.005
z_score = residual / sigma # Deja centre part l'hypothese de gauss-markov
print(f'z-score = {np.round(z_score,4)}')

if z_score > 2.0: # AUD est z_score plus eloigne de sa tendance (TROP CHER)
    print('VENTE du AUD/USD')
    print('ACHAT du NZD/USD')
elif z_score < -2.0:
    print('ACHAT du AUD/USD')
    print('VENTE du NZD/USD')
else:
    pass # NEUTRE

print('yo')


--- EXERCICE 2 -----
y_pred = 0.52
residual = 0.015
z-score = 3.0
VENTE du AUD/USD
ACHAT du NZD/USD


In [5]:

%load_ext autoreload
%autoreload 2
import os
import sys
import numpy as np

project_root = os.path.abspath('/home/user/perso/trading/alphalab')

if project_root not in sys.path:
    sys.path.append(project_root)

from enl.bivariate_model import ENLBivariateModel

# 1. Instanciation et Calibration Historique
model = ENLBivariateModel()
model.set_parameters(intercept=-0.02, coef=0.90, sigma_e=0.0050)

# 2. Flux d'entrée du marché Forex (Vecteurs NumPy)
nzd_usd = np.array([0.6000])  # Variable explicative X
aud_usd = np.array([0.5350])  # Variable dépendante Y

# 3. Exécution du pipeline
y_pred = model.predict(nzd_usd)
residuals = model.residuals(aud_usd, nzd_usd)
z_scores = model.z_scores(aud_usd, nzd_usd)

# 4. Diagnostics Console
print(f"y_pred   : {y_pred[0]:.4f}")
print(f"residual : {np.round(residuals[0], 4)}")
print(f"z-score  : {np.round(z_scores[0], 4)}")
print(f"Decision : {model.generate_signal(z_scores[0])}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
y_pred   : 0.5200
residual : 0.015
z-score  : 3.0
Decision : short
